# 15 · JSON & Serialization

APIs, config, logs and event streams are overwhelmingly **JSON**. Python's
`json` module maps cleanly between JSON text and Python objects (dicts, lists,
str, int, float, bool, None). This notebook also covers **JSON Lines** — the
dominant format for event data — and `pickle`.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## `loads`/`dumps` vs `load`/`dump`

- `json.loads(s)` / `json.dumps(obj)` work with **strings** (the `s` = string).
- `json.load(f)` / `json.dump(obj, f)` work with **files**.

JSON objects become dicts; arrays become lists.

In [ ]:
import json

text = '{"id": 7, "tags": ["vip", "eu"], "active": true, "note": null}'
obj = json.loads(text)
print(type(obj), obj)
print('tags:', obj['tags'], '| active:', obj['active'], '| note:', obj['note'])

back = json.dumps(obj, indent=2)
print(back)

## JSON Lines (`.jsonl`) — the event-stream format

One JSON object per line. It streams beautifully (parse a line at a time) and
appends cheaply — which is why clickstreams and logs use it. Our
`events.jsonl` is exactly this.

In [ ]:
import json

events = []
with open(RAW / 'events.jsonl', encoding='utf-8') as f:
    for line in f:
        events.append(json.loads(line))

print('events:', len(events))
print('first event:', events[0])

## Navigating nested payloads

Event records have a nested `payload` object with **optional** keys. Use
`.get()` with defaults so missing keys don't crash the pipeline.

In [ ]:
from collections import Counter

types = Counter(e['event_type'] for e in events)
print('event types:', dict(types))

# Only 'search' events carry payload['query']
queries = [e['payload'].get('query') for e in events if e['event_type'] == 'search']
print('top searches:', Counter(queries).most_common(3))

# Only 'add_to_cart' carries payload['qty']; default to 0 elsewhere
cart_qty = sum(e['payload'].get('qty', 0) for e in events)
print('total cart qty:', cart_qty)

## Writing JSON and JSON Lines

Serialize a summary to pretty JSON, and write records back out as `.jsonl`.

In [ ]:
import json
from collections import Counter

summary = {
    'total_events': len(events),
    'by_type': dict(Counter(e['event_type'] for e in events)),
}
out = DATA / 'staging' / 'event_summary.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(out.read_text(encoding='utf-8'))

## Non-serializable types & custom encoders

`json` doesn't know how to serialize a `datetime`, `set`, or `Decimal` out of
the box — it raises `TypeError`. Provide a `default=` function to convert them.

In [ ]:
import json
from datetime import datetime

record = {'id': 1, 'seen_at': datetime(2024, 6, 1, 9, 30)}

def encode(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f'not serializable: {type(obj)}')

print(json.dumps(record, default=encode))

## `pickle` — Python-native serialization (use with care)

`pickle` serializes almost any Python object to bytes. It's convenient for
caching intermediate results, but **never unpickle untrusted data** (it can
execute arbitrary code) and it isn't cross-language. Prefer JSON/Parquet for
interchange.

In [ ]:
import pickle

obj = {'model': 'v1', 'weights': [0.1, 0.2, 0.3], 'tags': {'a', 'b'}}
blob = pickle.dumps(obj)
print('pickled bytes:', len(blob))
restored = pickle.loads(blob)
print('restored:', restored)

### Recap

`loads/dumps` (strings) vs `load/dump` (files); JSON Lines streams event data;
navigate nested payloads defensively with `.get()`; supply `default=` for
`datetime`/`set`/`Decimal`; `pickle` is Python-only and unsafe on untrusted
input. Next: dates and times.